# 2장 예제 — 2.10 pandas 기초 — 표 데이터를 다루는 표준 도구

- **교재 출처**: manuscript/21_ch2_pandas_입출력.md
- **실행 방법**: example 폴더에서  python "2장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 07_2-7_함수.py, 08_2-8_모듈과_패키지.py, 09_2-9_예외_처리와_파일_입출력.py
- **실행 상태**: 단독 실행 확인됨(이 파일만 돌려도 끝까지 실행된다).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

### Series와 DataFrame `필수`

In [ ]:
import pandas as pd

s = pd.Series([3607.4, 3603.1, 3635.7],
              index=["L25000", "L25001", "L25002"], name="최종_용량_mAh")
print(s)

# 교재 실행 결과 ------------------------------------------------------
#   L25000    3607.4
#   L25001    3603.1
#   L25002    3635.7
#   Name: 최종_용량_mAh, dtype: float64

### read_csv로 데이터 불러오기 `필수`

In [ ]:
df = pd.read_csv("data/battery_process_data.csv")
print(df.head())
print(df.shape)

# 교재 실행 결과 ------------------------------------------------------
#      Lot_ID  믹싱_RPM  믹싱_온도  코팅_토출압력  ...  프레스_Gap  최종_용량_mAh  전극_면저항_mOhmcm2  불량_여부
#   0  L25000    1774   24.9    116.4  ...     93.5     3607.4          1428.4      0
#   1  L25001    1667   23.9    114.7  ...     93.2     3603.1          1444.9      0
#   2  L25002    1810   24.4    123.5  ...     90.8     3635.7          1445.3      0
#   3  L25003    1825   26.0    122.0  ...     97.6     3598.3          1545.2      0
#   4  L25004    1594   25.0    108.8  ...     96.4     3539.8          1483.1      0
#   [5 rows x 10 columns]
#   (1000, 10)

### info와 describe — 데이터 건강검진 `필수`

In [ ]:
df.info()
print(df.describe().round(2))

# 교재 실행 결과 ------------------------------------------------------
#   <class 'pandas.DataFrame'>
#   RangeIndex: 1000 entries, 0 to 999
#   Data columns (total 10 columns):
#    #   Column          Non-Null Count  Dtype  
#   ---  ------          --------------  -----  
#    0   Lot_ID          1000 non-null   str    
#    1   믹싱_RPM          1000 non-null   int64  
#    2   믹싱_온도           995 non-null    float64
#    3   코팅_토출압력         1000 non-null   float64
#    4   건조로_1구간_온도      1000 non-null   float64
#    5   프레스_압력          1000 non-null   float64
#    6   프레스_Gap         1000 non-null   float64
#    7   최종_용량_mAh       1000 non-null   float64
#    8   전극_면저항_mOhmcm2  1000 non-null   float64
#    9   불량_여부           1000 non-null   int64  
#   dtypes: float64(7), int64(2), str(1)
#   memory usage: 78.3 KB
#           믹싱_RPM   믹싱_온도  코팅_토출압력  ...  최종_용량_mAh  전극_면저항_mOhmcm2    불량_여부
#   count  1000.00  995.00  1000.00  ...    1000.00         1000.00  1000.00
#   mean   1747.68   24.88   120.27  ...    3599.00         1514.02     0.02

### 열 선택 `필수`

In [ ]:
print(df["최종_용량_mAh"].head(3))                          # 한 열 → Series
print(df[["Lot_ID", "최종_용량_mAh", "불량_여부"]].head(3))   # 여러 열 → DataFrame

# 교재 실행 결과 ------------------------------------------------------
#   0    3607.4
#   1    3603.1
#   2    3635.7
#   Name: 최종_용량_mAh, dtype: float64
#      Lot_ID  최종_용량_mAh  불량_여부
#   0  L25000     3607.4      0
#   1  L25001     3603.1      0
#   2  L25002     3635.7      0

### 행 필터링 — 불량 로트만 추출하기 `필수`

In [ ]:
defect = df[df["불량_여부"] == 1]
print(defect.shape)
print(defect[["Lot_ID", "최종_용량_mAh", "전극_면저항_mOhmcm2"]].head())
print(f"전체 불량률: {df['불량_여부'].mean() * 100:.2f}%")
cond = (df["프레스_압력"] > 48) & (df["불량_여부"] == 1)   # AND는 &, 조건마다 괄호
print(df[cond].shape[0], "개 로트")

# 교재 실행 결과 ------------------------------------------------------
#   (19, 10)
#        Lot_ID  최종_용량_mAh  전극_면저항_mOhmcm2
#   97   L25097     3515.4          1674.7
#   101  L25101     3531.1          1557.5
#   176  L25176     3531.4          1688.4
#   183  L25183     3501.0          1737.4
#   253  L25253     3532.0          1616.8
#   전체 불량률: 1.90%
#   1 개 로트

### 새 열 계산하기 `필수`

In [ ]:
df["용량_면저항_비"] = df["최종_용량_mAh"] / df["전극_면저항_mOhmcm2"]
print(df[["Lot_ID", "최종_용량_mAh", "전극_면저항_mOhmcm2", "용량_면저항_비"]].head(3).round(3))

# 교재 실행 결과 ------------------------------------------------------
#      Lot_ID  최종_용량_mAh  전극_면저항_mOhmcm2  용량_면저항_비
#   0  L25000     3607.4          1428.4     2.525
#   1  L25001     3603.1          1444.9     2.494
#   2  L25002     3635.7          1445.3     2.516

### groupby — 그룹별 집계 `필수`

In [ ]:
grp = df.groupby("불량_여부")[["최종_용량_mAh", "전극_면저항_mOhmcm2", "프레스_압력"]].mean()
print(grp.round(2))
agg = df.groupby("불량_여부")["최종_용량_mAh"].agg(["count", "mean", "std", "min", "max"])
print(agg.round(2))

# 교재 실행 결과 ------------------------------------------------------
#          최종_용량_mAh  전극_면저항_mOhmcm2  프레스_압력
#   불량_여부                                   
#   0        3600.51         1510.91   39.88
#   1        3521.17         1674.32   41.54
#          count     mean    std     min     max
#   불량_여부                                       
#   0        981  3600.51  32.71  3482.9  3724.7
#   1         19  3521.17  15.58  3497.6  3563.9

### to_csv — 결과 저장 `필수`

In [ ]:
defect_only = df[df["불량_여부"] == 1]
defect_only.to_csv("data/defect_lots.csv", index=False, encoding="utf-8-sig")
print("저장 완료:", len(defect_only), "행")

# 교재 실행 결과 ------------------------------------------------------
#   저장 완료: 19 행

### .loc와 .iloc — 이름으로, 번호로 고르기 `필수`

In [ ]:
print(df.iloc[0:3, 0:3])                                             # 번호로: 0~2행, 0~2열
print(df.loc[0:2, ["Lot_ID", "프레스_Gap", "전극_면저항_mOhmcm2"]])     # 이름으로
print(df.loc[df["불량_여부"] == 1, "전극_면저항_mOhmcm2"].mean().round(2))

# 교재 실행 결과 ------------------------------------------------------
#      Lot_ID  믹싱_RPM  믹싱_온도
#   0  L25000    1774   24.9
#   1  L25001    1667   23.9
#   2  L25002    1810   24.4
#      Lot_ID  프레스_Gap  전극_면저항_mOhmcm2
#   0  L25000     93.5          1428.4
#   1  L25001     93.2          1444.9
#   2  L25002     90.8          1445.3
#   1674.32

### NumPy 배열 — pandas를 떠받치는 엔진 `필수`

In [ ]:
import numpy as np

arr = np.array([1428.4, 1444.9, np.nan, 1545.2])
print(arr * 2)                       # 벡터 연산: 원소 전체에 한 번에
print(np.isnan(arr).sum(), "개 결측")
print(np.nanmean(arr).round(2))      # 결측을 빼고 평균
X = df[["프레스_Gap", "전극_면저항_mOhmcm2"]].to_numpy()
print(X.shape, X.mean(axis=0).round(2))   # axis=0은 열 방향(위→아래) 집계
print(np.corrcoef(X[:, 0], X[:, 1])[0, 1].round(4))

# 교재 실행 결과 ------------------------------------------------------
#   [2856.8 2889.8    nan 3090.4]
#   1 개 결측
#   1472.83
#   (1000, 2) [  95.3  1514.02]
#   0.8352